# TVC Thermoacoustic Regime Classification - Demo

This notebook demonstrates the key results without requiring raw data files. All results are loaded from cached files.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import os

# Load cached results
data = np.load('../results/features.npz', allow_pickle=True)
X = data['X_combined']           # (20, 93) feature matrix
y_5c = data['y']                  # 5-class labels
y_3c = data['y_3class']           # 3-class labels
LD = data['LD_values']            # L/D values
feat_names = list(data['feature_names_combined'])

preds = pd.read_csv('../results/all_predictions.csv')

REGIMES_5C = ['Limit Cycle', 'Period-2', 'Quasi-periodic', 'SNA', 'Chaos']
REGIMES_3C = ['Periodic', 'Quasi-periodic', 'Aperiodic']

print(f"Dataset: {X.shape[0]} recordings, {X.shape[1]} features")
print(f"5-class distribution: { {REGIMES_5C[i]: int((y_5c==i).sum()) for i in range(5)} }")
print(f"3-class distribution: { {REGIMES_3C[i]: int((y_3c==i).sum()) for i in range(3)} }")


## Dataset Overview

20 pressure recordings from a trapped vortex combustor, spanning cavity L/D from 0.75 to 2.625 at fixed Re=8000 and phi=0.72. Each recording has a 93-feature vector combining 66 windowed features (50 ms windows) and 27 nonlinear dynamics features (full-recording).


In [ ]:
# Train RF on all 20 samples (deployment mode)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X_scaled, y_5c)

# Top 10 features
imp = pd.DataFrame({'feature': feat_names, 'importance': rf.feature_importances_})
imp = imp.sort_values('importance', ascending=False).head(10)
print("Top 10 features by RF importance:")
print(imp.to_string(index=False))


In [ ]:
fig_dir = '../results/key_figures'
key_figs = [
    ('comparison_3class.png', '3-class accuracy across all 19 methods'),
    ('feature_importance.png', 'Feature importance (combined set)'),
    ('cm_3c_hard_vote.png', 'Hard Vote ensemble: 100% on 3-class'),
    ('gradcam_5c.png', 'Grad-CAM: what the 1D-CNN attends to'),
]
for fname, title in key_figs:
    path = os.path.join(fig_dir, fname)
    if os.path.exists(path):
        img = plt.imread(path)
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.imshow(img)
        ax.set_title(title, fontsize=12)
        ax.axis('off')
        plt.tight_layout()
        plt.show()


## Model Comparison Summary

Accuracy across all 19 methods on both 3-class and 5-class formulations. All values from leave-one-L/D-out cross-validation.


In [ ]:
# Compute accuracy for each method
pred_5c_cols = [c for c in preds.columns if c.startswith('pred_5c_')]
pred_3c_cols = [c for c in preds.columns if c.startswith('pred_3c_')]

print("=== 5-CLASS ACCURACY ===")
for col in sorted(pred_5c_cols):
    acc = (preds[col] == preds['true_5c']).mean() * 100
    name = col.replace('pred_5c_', '')
    print(f"  {name:35s}  {acc:.1f}%")

print("\n=== 3-CLASS ACCURACY ===")
for col in sorted(pred_3c_cols):
    acc = (preds[col] == preds['true_3c']).mean() * 100
    name = col.replace('pred_3c_', '')
    print(f"  {name:35s}  {acc:.1f}%")


## Seed Stability (Deep Learning)

| Model | Original | Mean +/- Std (5 seeds) | Verdict |
|-------|----------|----------------------|---------|
| 1D-CNN | 75% | 69% +/- 4.9% | Stable |
| LSTM | 55% | 52% +/- 4.0% | Stable |
| GRU | 55% | 56% +/- 3.7% | Stable |
| 2D-CNN (RP) | 80% | 69% +/- 9.7% | Seed-sensitive |


## Cross-Condition Generalization

Trained on phi=0.72 (20 recordings), tested on phi=0.61 (15 recordings):

- High-confidence recordings (5 LC + 1 Chaos): **100% accuracy**
- Full test set (3-class): **81%**, all errors on Quasi-periodic class
- Periodic and Aperiodic regimes transfer perfectly across conditions


In [ ]:
diag_figs = [
    ('training_curves_1dcnn.png', '1D-CNN training curves'),
    ('training_curves_2dcnn.png', '2D-CNN training curves'),
    ('per_sample_heatmap_5c.png', 'Per-sample correctness across methods'),
]
for fname, title in diag_figs:
    path = os.path.join(fig_dir, fname)
    if os.path.exists(path):
        img = plt.imread(path)
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.imshow(img)
        ax.set_title(title, fontsize=12)
        ax.axis('off')
        plt.tight_layout()
        plt.show()


## Summary

- 100% on 3-class with hard-voting ensemble (robust, verified)
- 69% +/- 9.7% on 5-class with 2D-CNN (seed-sensitive)
- Cross-channel coherence is the top regime discriminator
- Features generalize across operating conditions (phi=0.72 to phi=0.61)
- Classification is sample-limited at 20 recordings, not feature-limited
